## 基础知识

#### State Message

In [3]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END, MessagesState

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-flash",
    extra_body={"thinking": {"type": "disabled"}}
)

class OverallState(MessagesState):
    username: str
    output: str

def node_a(state: OverallState) -> OverallState:
    return {
        "messages": [HumanMessage("你好，我是" + state["username"])],
    }

def node_llm(state: OverallState) -> OverallState:
    response = model.invoke(state["messages"])
    return {
        "messages": [response],
        "output": response.content,
    }


builder = StateGraph(OverallState)
builder.add_node(node_a)
builder.add_node(node_llm)
builder.add_edge(START, 'node_a')
builder.add_edge('node_a', 'node_llm')
builder.add_edge('node_llm', END)
graph = builder.compile()
result = graph.invoke({"username": "Max"})
print(result["output"])


你好，Max！很高兴认识你。我是DeepSeek，由深度求索公司创造的AI助手。

有什么我可以帮你的吗？无论是回答问题、协助写作、分析信息，还是聊聊天，我都很乐意帮忙！
